In [2]:
import cv2
import numpy as np
import os
from datetime import datetime

In [5]:


VIDEO_SOURCE = "video.mp4"
OUTPUT_VIDEO = "output_motion.mp4"

MIN_CONTOUR_AREA = 7000       
DIFF_THRESHOLD = 15         
MIN_MOTION_PIXELS = 10000     

os.makedirs("Detections", exist_ok=True)

cap = cv2.VideoCapture(VIDEO_SOURCE)
if not cap.isOpened():
    raise RuntimeError("Video could not be opened")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps is None or fps <= 1:
    fps = 25.0

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))#get video prop

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

def preprocess(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) #simpler, faster
    gray = cv2.GaussianBlur(gray, (15, 15), 0) #reduce noise and tiny pixel changes
    return gray

ret, prev_frame = cap.read()
if not ret:
    raise RuntimeError("Could not read first frame")

prev_gray = preprocess(prev_frame)
kernel = np.ones((5, 5), np.uint8)

print("Processing video... press Q to stop")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    curr_gray = preprocess(frame)
    diff = cv2.absdiff(prev_gray, curr_gray)

    _, motion = cv2.threshold(diff, DIFF_THRESHOLD, 255, cv2.THRESH_BINARY) #pixels with difference >= 20 become white (255)
    motion = cv2.morphologyEx(motion, cv2.MORPH_OPEN, kernel, iterations=2)
    motion = cv2.dilate(motion, kernel, iterations=2)   

    motion_pixels = cv2.countNonZero(motion)            

    contours, _ = cv2.findContours(motion, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    display = frame.copy()
    active = False

    
    if motion_pixels >= MIN_MOTION_PIXELS: #DECISSION
        active = True

    for c in contours:
        if cv2.contourArea(c) > MIN_CONTOUR_AREA:
            active = True
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(display, (x, y), (x+w, y+h), (0, 255, 255), 2)

            crop = frame[y:y+h, x:x+w]
            name = datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ".jpg"
            cv2.imwrite("Detections/" + name, crop)

    if active:
        cv2.putText(display, f"MOTION DETECTED (pixels={motion_pixels})", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
        out.write(display)   
    else:
        cv2.line(display, (0, 0), (width, height), (0, 0, 255), 4)
        cv2.line(display, (width, 0), (0, height), (0, 0, 255), 4)
        cv2.putText(display, f"BLANK (pixels={motion_pixels})", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

    cv2.imshow("Motion Detection", display)
    

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    prev_gray = curr_gray

cap.release()
out.release()
cv2.destroyAllWindows()

print("Done!")
print("Output video saved as:", OUTPUT_VIDEO)
print("Detections saved in folder: Detections/")


Processing video... press Q to stop
Done!
Output video saved as: output_motion.mp4
Detections saved in folder: Detections/
